# TrimCI-Flow — DMET 1-shot (non-overlapping)

This notebook runs 1-shot non-overlapping DMET on Fe4S4 using TrimCI as the impurity solver.

**Scientific role:** compute the DMET total energy as a sum of fragment contributions obtained from  
Schmidt-decomposed impurity problems, each solved with TrimCI.

**Pipeline per fragment:**
1. RHF → mean-field 1-RDM `γ_MF`  
2. Schmidt decomposition → bath orbitals + frozen core  
3. Project full-system `h1`, `eri` onto impurity basis (frag + bath)  
4. Add Fock correction from frozen core → `h1_solver`  
5. TrimCI solve on 24-orbital impurity  
6. Extract 1-RDM (`γ`) and 2-RDM (`Γ2`), verify convention on fragment 0  
7. Fragment energy formulas A (debug), B (primary), C (democratic)  

**Energy formulas:**
- `E_B = Σ_I [ h1_solver[frag,imp]·γ[frag,imp] + 0.5·eri_chem[frag,imp,imp,imp]·Γ2[frag,imp,imp,imp] ]`  
  where eri contraction is `(pr|qs)·Γ2[p,q,r,s]` (physicist convention)

**Known result (threshold=0.06):**  
- `E_DMET_B = −165.940114 Ha`  
- Reference (brute-force TrimCI) = `−327.1920 Ha`  
- Gap = `+161.25 Ha` — expected for 1-shot non-self-consistent DMET

**Runtime:** ~3–5 min (3 × TrimCI impurity solve, 24 orbitals each)  
Run inside `qflowenv` at `/home/unfunnypanda/Proj_Flow/qflowenv/`.

In [ ]:
# Cell 1 — Environment and path setup
import sys, os

PROJECT_ROOT = os.path.expanduser('~/Proj_Flow')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')
print(f'Python: {sys.version}')

In [ ]:
# Cell 2 — Imports
import json
import datetime
from pathlib import Path
import numpy as np

from TrimCI_Flow.dmet import run_dmet_1shot

print('Imports OK')

In [ ]:
# Cell 3 — Configuration
ROOT = Path.cwd()

FCIDUMP = (
    ROOT
    / 'Fe4S4_251230orbital_-327.1920_10kdets'
    / 'Fe4S4_251230orbital_-327.1920_10kdets'
    / 'fcidump_cycle_6'
)

TRIMCI_CONFIG = {
    'threshold':            0.06,   # CI selection threshold (Ha)
    'max_final_dets':       'auto', # ~50 dets for 24-orb impurity
    'max_rounds':           2,
    'num_runs':             1,
    'pool_build_strategy':  'heat_bath',
    'verbose':              False,
}

timestamp  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = ROOT / 'TrimCI_Flow' / 'Outputs' / 'dmet' / f'outs_notebook_dmet_{timestamp}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'FCIDUMP  : {FCIDUMP}')
print(f'Config   : {TRIMCI_CONFIG}')
print(f'Output   : {OUTPUT_DIR.relative_to(ROOT)}/')

In [ ]:
# Cell 4 — Run DMET  (3–5 min; TrimCI output streams to stdout)
print(f'Starting DMET 1-shot at {datetime.datetime.now().isoformat()}')

result = run_dmet_1shot(
    fcidump_path  = str(FCIDUMP),
    trimci_config = TRIMCI_CONFIG,
    output_dir    = str(OUTPUT_DIR),
)

print(f'\nFinished at {datetime.datetime.now().isoformat()}')

In [ ]:
# Cell 5 — Energy summary
REFERENCE = -327.1920   # brute-force TrimCI (10 095 dets)

print('=' * 60)
print('  DMET 1-shot — Energy Summary')
print('=' * 60)
print(f'  E_HF                           = {result.E_hf:.6f} Ha')
print(f'  E_DMET_A (1-body, debug)       = {result.E_dmet_a:.6f} Ha')
print(f'  E_DMET_B (2-RDM, primary)      = {result.E_dmet:.6f} Ha  <- canonical')
print(f'  E_DMET_C (democratic, diag.)   = {result.E_dmet_c:.6f} Ha')
print(f'  Reference (brute-force TrimCI) = {REFERENCE:.4f}    Ha')
print(f'  Error (B − reference)          = {result.E_dmet - REFERENCE:+.4f} Ha')
print('=' * 60)
print(f'  Total DMET dets : {result.total_dets}  (baseline: 118, brute-force: 10 095)')

In [ ]:
# Cell 6 — Fragment breakdown
print(f'{"Frag":>4}  {"n_orbs":>6}  {"n_dets":>6}  {"E_imp (Ha)":>14}  orbs[:4]')
print('-' * 55)
for i, (orbs, nd, eimp) in enumerate(
    zip(result.fragment_orbs, result.fragment_n_dets, result.fragment_energies)
):
    print(f'  F{i}  {len(orbs):>6}  {nd:>6}  {eimp:>14.6f}  {orbs[:4]}...')

In [ ]:
# Cell 7 — Save extended metadata (results.json written by run_dmet_1shot already)
metadata = {
    'run_type':          'dmet_1shot_notebook',
    'timestamp':         datetime.datetime.now().isoformat(),
    'fcidump':           str(FCIDUMP),
    'trimci_config':     TRIMCI_CONFIG,
    'E_hf':              result.E_hf,
    'E_dmet_a':          result.E_dmet_a,
    'E_dmet_b':          result.E_dmet,
    'E_dmet_c':          result.E_dmet_c,
    'reference_Ha':      REFERENCE,
    'error_b_Ha':        result.E_dmet - REFERENCE,
    'total_dets':        result.total_dets,
    'fragment_n_dets':   result.fragment_n_dets,
    'fragment_orbs':     result.fragment_orbs,
    'fragment_energies_imp': result.fragment_energies,
}

(OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')
print('Saved:')
for fn in ['results.json', 'run_metadata.json']:
    print(f'  {(OUTPUT_DIR / fn).relative_to(ROOT)}')

In [ ]:
# Cell 8 — Load and display saved metadata
with (OUTPUT_DIR / 'run_metadata.json').open() as f:
    saved = json.load(f)

print('=== Saved run_metadata.json ===')
for k, v in saved.items():
    print(f'  {k}: {v}')

## Interpretation

**1-shot non-overlapping DMET on Fe4S4** (36 orbitals, 54 electrons).  
Three 12-orbital fragments selected by ascending h1-diagonal energy.

| Metric | Value |
|---|---|
| E_DMET_B (primary) | −165.940114 Ha |
| E_HF baseline | −325.999622 Ha |
| Reference (brute-force TrimCI) | −327.1920 Ha |
| Error vs reference | +161.25 Ha |
| Total impurity dets | 150 (3 × 50) |

**Why is the error large?**

- **1-shot DMET** uses the mean-field `γ_MF` as the bath-construction reference without  
  self-consistently updating it — the bath does not capture inter-fragment correlation.  
- RHF does not converge for Fe4S4 (strongly correlated system), so `γ_MF` is unreliable  
  from the start.
- The non-overlapping partition (12+12+12) discards all inter-fragment two-electron terms.
- `threshold=0.06` is aggressive — each impurity is solved with only ~50 determinants.

**E_DMET_A** (−679.68 Ha) is unphysically low because it uses `h1_phys` (no Fock correction),  
double-counting the bath–fragment interaction. This pathology confirms the Fock correction  
in `h1_solver` is non-trivial.

**2-RDM convention check** passed on fragment 0 with discrepancy < 10⁻¹³ Ha,  
confirming the physicist ERI transpose is correct.

**Next step:** self-consistent DMET (iterate `γ_MF` ← fragment 1-RDMs until convergence).

## How to run again

### From terminal
```bash
cd /home/unfunnypanda/Proj_Flow
source qflowenv/bin/activate
python TrimCI_Flow/dmet/runners/run_dmet_1shot.py
```

### From this notebook
```bash
cd /home/unfunnypanda/Proj_Flow
source qflowenv/bin/activate
jupyter notebook TrimCI_Flow/Flow_DMET.ipynb
# Kernel → Restart & Run All
```

### To change TrimCI threshold
Edit `TRIMCI_CONFIG['threshold']` in Cell 3.  
Lower threshold → more determinants → more accurate (but slower).

### Expected output files
- `TrimCI_Flow/Outputs/dmet/outs_notebook_dmet_<timestamp>/results.json`
- `TrimCI_Flow/Outputs/dmet/outs_notebook_dmet_<timestamp>/run_metadata.json`

### Key result
`result.E_dmet` (= `E_DMET_B`) is the canonical DMET energy.  
`result.E_dmet_a` and `result.E_dmet_c` are diagnostic only.